# 批处理

到上一章为止，我们每次迭代都是用一个样本完成。这种方式称为**随机梯度下降**（Stochastic Gradient Descent，SGD），虽然简单，但有一个明显缺点：**每次只处理一个样本，无法利用 NumPy 的向量化并行计算能力**，在大规模数据上效率低下。

实际上，我们完全可以一次将多个样本同时送入模型，让矩阵运算一步完成所有计算。这种方式称为**批处理**（Batch Processing），是现代深度学习的标准做法。

按照每次处理的样本数量，梯度下降分为三类：

* **随机梯度下降**（SGD）：每次处理 **1 个样本**，逐一迭代；
* **全批量梯度下降**（Batch GD）：每次处理**全部样本**，一次迭代；
* **小批量梯度下降**（Mini-batch GD）：每次处理**数个样本**，是目前最常用的方式。

## 收敛

**收敛**（Convergence）指损失值在训练过程中不再显著下降，模型参数趋于稳定。

三种方式的收敛特性各有不同：

* **随机梯度下降**：每次只从一个样本学习，梯度方向随机性大，收敛路径曲折震荡，但每轮参数更新次数最多；
* **全批量梯度下降**：每次利用全部数据计算梯度，方向最稳定，单步更新质量高，但每轮只更新一次；
* **小批量梯度下降**：折中方案。比随机梯度下降稳定，比全批量灵活，兼顾效率与稳定性。

## 泛化

**泛化**（Generalization）衡量模型在从未见过的新数据（测试集）上的表现：

* **随机梯度下降**：梯度噪声反而有助于跳出局部极小值，泛化能力通常更好；
* **全批量梯度下降**：梯度方向过于精确，容易陷入局部极小值，在复杂模型中泛化能力相对较弱；
* **小批量梯度下降**：适度的梯度噪声，在收敛速度和泛化能力之间取得平衡。

``💡 批处理方式对泛化的影响，在非线性模型（后续章节）中更为明显。``

## 拟合

**拟合**（Fitting）指模型在训练集上学习数据规律的程度，是泛化能力的基础。拟合不理想通常有两种情况：

* **欠拟合**（Underfitting）：训练集和测试集上表现都差。通常是训练不足，或模型过于简单；
* **过拟合**（Overfitting）：训练集上接近完美，测试集上却表现糟糕。通常是模型过于复杂，死记硬背了训练数据中的噪声，而不是学到真正的规律。

``💡 小批量梯度下降兼顾收敛速度与泛化能力，在实践中是最常用的训练方式。``

In [1]:
import numpy as np

## 数据集

与上一章相同的数据集：4 个训练样本，1 个测试样本。

### 训练数据：特征、标签

In [2]:
train_features = np.array([[22.5, 72.0],
                           [31.4, 45.0],
                           [19.8, 85.0],
                           [27.6, 63.0]])
train_labels = np.array([[95],
                         [210],
                         [70],
                         [155]])

### 测试数据：特征、标签

In [3]:
test_features = np.array([[28.1, 58.0]])
test_labels = np.array([[165]])

## 模型

本章对**梯度函数**和**反向函数**做了改动以支持批处理，其余与上一章相同。

### 参数：权重、偏置

In [4]:
weight = np.ones((1, 2)) / 2
bias = np.zeros(1)

### 推理函数

In [5]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

In [6]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

### 梯度函数

批处理时，一次有 $n$ 个样本同时参与计算。完整的均方误差公式是：

$$
L = \frac{1}{n} \sum_{i=1}^{n} (y_i - p_i)^2
$$

对预测值 $p$ 求导，误差项变为：

$$
\delta_i = -\frac{2}{n}(y_i - p_i)
$$

与单样本相比，多了一个 $\frac{1}{n}$ 因子。这个因子的作用是**对批次内的梯度取平均**，防止批次越大、梯度越大，造成训练不稳定。

In [7]:
def gradient(p, y):
    return -2 * (y - p) / len(y)

### 反向函数

批处理的梯度函数会为每个样本计算一个误差项，形成形状为 `(批大小, 1)` 的矩阵 $\delta$。

反向函数需要把这批误差项汇总成一次参数更新：

**权重梯度**：完整公式是：

$$
\frac{\partial L}{\partial w} = \sum_i \delta_i \cdot x_i
$$

用矩阵乘法可以一步完成：

$$
\Delta w = \delta^\top \cdot X
$$

其中 $\delta^\top$ 形状为 `(1, 批大小)`，$X$ 形状为 `(批大小, 特征数)`，结果为 `(1, 特征数)`，与权重矩阵形状一致。

**偏置梯度**：对批次内所有误差项求和：

$$
\Delta b = \sum_i \delta_i
$$

In [8]:
def backward(x, d, w, b, lr):
    w -= d.T @ x * lr
    b -= np.sum(d, axis=0) * lr
    return w, b

## 训练

### 超参数：学习率

In [9]:
LEARNING_RATE = 0.00001

### 超参数：批大小

**批大小**（Batch Size）是本章引入的第二个超参数，定义了每次迭代同时处理的样本数量。

通过调整批大小，可以在同一个代码框架下实现三种梯度下降方式：批大小为 1 是随机梯度下降，批大小等于训练集总量是全批量，批大小介于两者之间是小批量。

``💡 实践中批大小通常选 2 的幂次方（16、32、64、128 ...），这不是数学要求，而是为了最大化电脑的并行计算效率。``

In [10]:
BATCH_SIZE = 2

### 迭代

批大小为 2 时，4 个训练样本被分为 2 个批次，全部只需要 2 次迭代（随机梯度下降需要 4 次）。

In [11]:
for i in range(0, len(train_features), BATCH_SIZE):
    # 每次读入一个批次
    features = train_features[i: i + BATCH_SIZE]
    labels = train_labels[i: i + BATCH_SIZE]

    # 一次前向传播处理整个批次
    predictions = forward(features, weight, bias)
    # 计算批次内每个样本的平均误差项
    delta = gradient(predictions, labels)
    # 将批次梯度汇总后更新参数
    weight, bias = backward(features, delta, weight, bias, LEARNING_RATE)

print(f'weight: {weight}')
print(f'bias:   {bias}')

weight: [[0.59388172 0.68104165]]
bias:   [0.00327249]


## 验证

### 推理

In [12]:
predictions = forward(test_features, weight, bias)
print(f'predictions: {predictions}')

predictions: [[56.19176426]]


### 评估

In [13]:
loss = mse_loss(predictions, test_labels)
print(f'loss: {loss:.4f}')

loss: 11839.2322


小批量（批大小 = 2，2 次更新）训练的损失值为 ``11,839``，而上一章随机梯度下降（批大小 = 1，4 次更新）的损失值为 ``9,563``。

这并不说明小批量"收敛更差"，差异来自**更新次数不同**：随机梯度下降迭代了 4 次，小批量只迭代了 2 次。

## 总结

| 方式               | 批大小 |    更新次数    | 梯度稳定性 | 计算效率 | 典型场景 |
|------------------|:------:|:----------:|:--------:|:------:|--------|
| 随机梯度下降           | 1 | 最多（$n$ 次）  | 低（噪声大）| 低 | 在线学习、数据流 |
| 小批量梯度下降 | 数个 |中等（$n/m$ 次） | 中等 | 高 | **深度学习标准做法** |
| 全批量梯度下降 | 全部 |  最少（1 次）   | 高（最稳定）| 极低（内存受限）| 小数据集 |

``💡 现代深度学习框架（PyTorch、TensorFlow）默认使用小批量梯度下降，批大小通常在 32 到 256 之间。``

## 课后练习

试一试全批量梯度下降训练（批大小 = 4，1 次更新），损失值有什么变化？